# Homework 1 — A refund cap

**Salih Tutun, PhD** · DAT 5668 · Module 1

Individual. Open this file in **Google Colab** (or run it locally). When you are done, download the `.ipynb` and upload it on Canvas.

## If you have never seen Agentic AI

A **chatbot** only writes a paragraph. An **agent** sits on a desk, looks at the world, **does** something, and then lives with what happened.

In this course that loop has four names:

| Everyday word | Course word | In this homework |
| --- | --- | --- |
| What can I see? | state $s$ | an order id, then the order record and the policy |
| What do I do? | action $a$ | `refund`, `refuse`, or `escalate` |
| The desk / the store | environment | `lookup_order`, `lookup_policy`, `open_case` |
| What came back? | reward $r$ and next state $s'$ | cash that left, or a human case in the queue |

You are writing the **decide** step: given an order id, pick the action. The tools below already observe the order and the policy. You do not train a model. You do not call ChatGPT. You write a small Python function that a refund desk could run every morning.

This is still the **LLM-based agent** family from Session 2: observe with tools, then act. Homework 1 only adds one control the CFO asked for.

## How to run this in Colab

1. Open the notebook from the Canvas link (GitHub → Open in Colab), or from [this Colab file](https://colab.research.google.com/github/salihtutun/Agentic-AI_Autonomous_Decision_Making/blob/main/Students/Homework/HW1-Refund-Cap.ipynb).
2. Click the first code cell (the tools). Press **Shift+Enter**. You should see `Tools are loaded. Next: write decide.`
3. In the `decide` cell, replace the `TODO` lines with real Python. Do not delete the numbered comments — they are the recipe.
4. Run the `decide` cell (Shift+Enter).
5. Type one sentence in `NOTE` (the next cell). Run that cell.
6. Run the last cell (`check()`). Read every line. `NOT YET` means go back to `decide` or `NOTE`. Stop when the last line says **Ready to submit.**
7. File → Download → `.ipynb`. Upload that file on the Canvas assignment.

If a cell says `NameError`, you skipped a cell above it. Run from the top.

## The business change (this is the whole assignment)

Session 2's agent refunds any eligible damaged order, even a **$220** screen. The CFO will not let software empty the till on a large ticket.

**New rule:** if the damage is eligible **and** `order["total"]` is **greater than** `CAP` (150), do **not** refund. Escalate, set amount to 0, and call `open_case` so a person owns the next state.

The mug at $80 is still an auto-refund. A change-of-mind is still a refuse. A missing order id is still an escalate. Only the **large eligible** refund is new.

Do **not** write `if order_id == "1001"`. An agent is a rule that still works on tomorrow's ticket. `check()` will invent two extra lamps to catch hard-coded ids.

## What you turn in

1. This notebook, `decide` filled in, `check()` printing **Ready to submit.**
2. `NOTE`: one sentence, at least 40 characters. Prompt: a $220 cracked screen is still damage — why does a person still have to release it?


## This cell — the desk (already written)

Run it. Do not edit it. These are the **tools**. An agent is allowed to call them. A chatbot that guesses \$999 is not allowed to skip them.

- `ORDERS` is the system of record. Keys are order ids.
- `lookup_order` is observe: given an id, return the record or `None`.
- `lookup_policy` is observe: given the customer note, return whether damage is eligible and the max age (30 days).
- `open_case` is act: put a human ticket on the queue. That is how escalate changes $s'$.
- `CAP = 150` is the new control. Use this name. Do not type the number 150 in five places.


In [ ]:
# RUN WITH ME. Do not edit this cell. Shift+Enter once, then leave it alone.
# Instructional Lumen Home desk — not a real firm's data.

ORDERS = {
    "1001": {"age_days": 12, "total": 80, "note": "cracked mug"},       # Maya: damage, small
    "1002": {"age_days": 8, "total": 45, "note": "changed my mind"},    # Jordan: not damage
    "1003": {"age_days": 10, "total": 220, "note": "cracked screen"},   # Priya: damage, OVER the cap
    "1004": {"age_days": 4, "total": 15, "note": "missing part"},       # small damage
}
CASES = []   # human queue. escalate must append here via open_case.
CAP = 150    # CFO rule: auto-refund only at or below this amount.


def lookup_order(order_id):
    """Observe the order. Unknown ids return None — that is a missing state."""
    return ORDERS.get(str(order_id))


def lookup_policy(note):
    """Observe the refund rule from the customer note. Crack or missing part = eligible damage."""
    note = note.lower()
    if "crack" in note or "missing" in note:
        return {"eligible_reason": True, "max_age_days": 30}
    return {"eligible_reason": False, "max_age_days": 0}


def open_case(order_id, reason):
    """Act: hand the ticket to a person. Always call this when you escalate."""
    CASES.append({"order_id": str(order_id), "reason": reason})
    return {"case_id": len(CASES), "reason": reason}


print("Tools are loaded. Next: write decide.")
print("Orders on the desk:", list(ORDERS.keys()), "| cap $", CAP)


## This cell — YOU WRITE `decide`

`decide(order_id)` must return a dictionary with **four keys**:

| Key | Meaning | Typical values |
| --- | --- | --- |
| `action` | what the desk does | `"refund"`, `"refuse"`, or `"escalate"` |
| `amount` | cash that leaves | `order["total"]` on a refund, otherwise `0` |
| `reason` | a short English sentence | anything honest; the checker does not grade the wording |
| `case_id` | the human ticket | `open_case(...)["case_id"]` when you escalate, else `None` |

Walk the numbered comments below, in order. Uncomment and finish each line. Copy from Session 2 if you have it, then add **step 5** (the cap). Amount is always `order["total"]` or `0`. Never type `999`.


In [ ]:
# YOU WRITE THIS. Search for TODO. Keep the comments. They are the recipe.

def decide(order_id):
    """Agent decide step: observe with tools, then act. Homework 1 adds the $150 cap."""
    # 1. OBSERVE the order.
    #    order = lookup_order(order_id)

    # 2. If the order is missing (order is None): a person must take it.
    #    case = open_case(order_id, "Unknown order.")
    #    return {"action": "escalate", "amount": 0, "reason": case["reason"], "case_id": case["case_id"]}

    # 3. OBSERVE the policy from the customer note.
    #    rule = lookup_policy(order["note"])
    #    eligible = rule["eligible_reason"] and order["age_days"] <= rule["max_age_days"]

    # 4. HOMEWORK 1 (new). Eligible AND over the cap: do not auto-refund.
    #    if eligible and order["total"] > CAP:
    #        case = open_case(order_id, "Over the auto-refund cap.")
    #        return {"action": "escalate", "amount": 0, "reason": case["reason"], "case_id": case["case_id"]}

    # 5. Eligible and inside the cap: refund the order total (not 999, not a made-up number).
    #    if eligible:
    #        return {"action": "refund", "amount": order["total"], "reason": "Inside policy and cap.", "case_id": None}

    # 6. Otherwise refuse (change of mind, too old, not damage). Amount 0. No case.
    #    return {"action": "refuse", "amount": 0, "reason": "Policy does not allow it.", "case_id": None}

    # TODO: delete this placeholder after the steps above are real code.
    return {"action": None, "amount": None, "reason": "TODO", "case_id": None}


print("decide is defined. Next: write NOTE, then run check().")


## This cell — YOU WRITE `NOTE`

One sentence in your own words. At least 40 characters. The checker only counts length, but we read it.

Prompt: Priya's screen is cracked and only 10 days old, so the **damage** rule would refund. It is \$220, so the **cap** stops the software. Why should a person still click release?


In [ ]:
# YOU TYPE HERE. Replace the empty string. Keep the quotes.
NOTE = ""  # TODO: one sentence, at least 40 characters.

print("NOTE length:", len(NOTE.strip()), "(need 40+)")


## This cell — the checks. Run with me. Do not edit.

`NOT YET` is information, not a grade. Fix `decide` or `NOTE`, re-run **those** cells, then re-run this one.

Named tickets: Maya 1001, Jordan 1002, Priya 1003 (now inside the 30-day window but over \$150), small damage 1004, missing id 9999.

Then the checker adds two lamps you have never seen. If those fail, you hard-coded order ids.


In [ ]:
# RUN WITH ME. Do not edit. Re-run after you change decide or NOTE.

def check():
    CASES.clear()
    named = {
        "1001": ("refund", 80),
        "1002": ("refuse", 0),
        "1003": ("escalate", 0),
        "1004": ("refund", 15),
        "9999": ("escalate", 0),
    }
    ok = True
    for order_id, (action, amount) in named.items():
        got = decide(order_id)
        match = got["action"] == action and got["amount"] == amount
        ok = ok and match
        print("PASS" if match else "NOT YET", "order", order_id)

    opened_cap = any(case["order_id"] == "1003" for case in CASES)
    opened_unknown = any(case["order_id"] == "9999" for case in CASES)
    print("PASS cap opened a case" if opened_cap else "NOT YET the over-cap ticket must call open_case")
    print("PASS unknown opened a case" if opened_unknown else "NOT YET unknown id must call open_case")

    # Unnamed tickets: an agent is a rule, not a list of ids from class.
    ORDERS["1099"] = {"age_days": 6, "total": 40, "note": "cracked lamp"}
    ORDERS["1098"] = {"age_days": 6, "total": 200, "note": "cracked lamp"}
    extra_ok = decide("1099")["action"] == "refund" and decide("1099")["amount"] == 40
    extra_ok = extra_ok and decide("1098")["action"] == "escalate" and decide("1098")["amount"] == 0
    print("PASS new tickets follow the same rule" if extra_ok else "NOT YET do not hard-code order ids")
    del ORDERS["1099"], ORDERS["1098"]

    note_ok = len(NOTE.strip()) >= 40
    print("PASS NOTE" if note_ok else "NOT YET write NOTE (40+ characters)")
    ready = ok and opened_cap and opened_unknown and extra_ok and note_ok
    print("Ready to submit." if ready else "Not ready.")


check()
